In [5]:
from pathlib import Path
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt


#loading initial imputs and outputs 
X= np.load('../data/initial_data/function_3/initial_inputs.npy')
y = np.load('../data/initial_data/function_3/initial_outputs.npy')

#checking shape of data 
print('Inputs Shape:', X.shape)
print('Outputs Shape:', y.shape)

#creating a dataframe 
df=pd.DataFrame(X, columns=['x1', 'x2', 'x3'])
df['y']=y

# --- add new observation for Function 3 ---
new_row = pd.DataFrame([{
    'x1': 0.42,
    'x2': 0.01,
    'x3': 0.50,
    'y': -0.076583}, {'x1': 0.892, 'x2': 0.99, 'x3': 0.01, 'y': -0.1275231}, {'x1': 0.99, 'x2': 0.9704, 'x3': 0.9508, 'y': -0.2542284},  {
        'x1': 0.4216,
        'x2': 0.3824,
        'x3': 0.5196,
        'y': -0.017678025
    }, {
        'x1': 0.99,
        'x2': 0.4608,
        'x3': 0.6176,
        'y': -0.0735932
    }, {
        'x1': 0.5980,
        'x2': 0.6176,
        'x3': 0.5980,
        'y': -0.0475605
    }])

# append to df
df = pd.concat([df, new_row], ignore_index=True)

# rebuild X and y from the *updated* df
X = df[['x1', 'x2', 'x3']].to_numpy()
y = df['y'].to_numpy()

# sanity check
print("len(df):", len(df))
print("Inputs Shape:", X.shape)
print("Outputs Shape:", y.shape)
print(df.tail())


#checking DataFrame
df.head(20)



Inputs Shape: (15, 3)
Outputs Shape: (15,)
len(df): 21
Inputs Shape: (21, 3)
Outputs Shape: (21,)
        x1      x2      x3         y
16  0.8920  0.9900  0.0100 -0.127523
17  0.9900  0.9704  0.9508 -0.254228
18  0.4216  0.3824  0.5196 -0.017678
19  0.9900  0.4608  0.6176 -0.073593
20  0.5980  0.6176  0.5980 -0.047560


,x1,x2,x3,y
0,0.171525,0.343917,0.248737,-0.112122
1,0.242114,0.644074,0.272433,-0.087963
2,0.534906,0.398501,0.173389,-0.111415
3,0.492581,0.611593,0.340176,-0.034835
4,0.134622,0.219917,0.458206,-0.048008
5,0.345523,0.941360,0.269363,-0.110621
6,0.151837,0.439991,0.990882,-0.398926
7,0.645503,0.397143,0.919771,-0.113869
8,0.746912,0.284196,0.226300,-0.131461
9,0.170477,0.697032,0.149169,-0.094190


In [20]:
# setting parameters and making 2D grid 
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

#hyperparamters for GP surrogate model and acqusition function 
matern_lengthscale = 0.1
nu = 2.5 #controls bumpiness - good default - higher = smoother 
noise_assumption = 1e-6 #gp will reestiamte 
xi = 0.04
#building a 2D grid to represent the test space 
#linspace(start, stop, num)
g = np.linspace(0.01, 0.99, 51)   
x1g, x2g, x3g = np.meshgrid(g, g, g)
X_grid = np.column_stack([x1g.ravel(), x2g.ravel(), x3g.ravel()])

In [22]:
#fit the gp to the current data 
kernel = (C(1.0, (1e-2, 1e2)) * Matern(length_scale=0.2, length_scale_bounds=(0.05, 5), nu=2.5) 
    + WhiteKernel(noise_level=0.3, noise_level_bounds=(1e-8, 1)))
gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=8, random_state=0)
gp.fit(X, y)

GaussianProcessRegressor(kernel=1**2 * Matern(length_scale=0.2, nu=2.5) + WhiteKernel(noise_level=0.3),
                         n_restarts_optimizer=8, normalize_y=True,
                         random_state=0)

In [24]:
from scipy.stats import norm
import numpy as np

def expected_improvement(mu, sigma, y_best, xi=0.02):
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

# predict GP on candidate set
mu, std = gp.predict(X_grid, return_std=True)

# best observed so far (maximization)
y_best = float(np.max(y))

# compute EI and select next point
ei = expected_improvement(mu, std, y_best, xi=xi)  # use your xi
ix = int(np.argmax(ei))
x_next = X_grid[ix]
print(f"x_next = ({x_next[0]:.6f}, {x_next[1]:.6f}, {x_next[2]:.6f})")

x_next = (0.421600, 0.794000, 0.010000)


In [91]:
#x_next check 
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances

y_best = float(np.max(y))
model        = gp
X_candidates = np.array(X_grid)   # shape (n_candidates, n_dims)
acq_values   = np.array(ei)   # shape (n_candidates,)
x_next       = np.array(x_next)   # shape (n_dims,)
y_best       = y_best
X_sampled    = np.array(X)   # all evaluated X so far


def _percentile_of_score(values, score):
    """Simple percentile rank: percentage of values <= score."""
    values = np.asarray(values).ravel()
    return float(100.0 * (np.sum(values <= score) / max(len(values), 1)))

def _first_length_scale_from_kernel(kernel):
    """Extract effective GP length-scale (handles RBF, Matern, etc.)."""
    if hasattr(kernel, "length_scale"):
        ls = getattr(kernel, "length_scale")
        try:
            ls = float(np.mean(ls))
        except Exception:
            ls = float(ls)
        return ls
    for attr in ("k1", "k2"):
        if hasattr(kernel, attr):
            ls = _first_length_scale_from_kernel(getattr(kernel, attr))
            if ls is not None:
                return ls
    return None

def _nearest_dist_over_ell(x_next, X_sampled, ell):
    """Compute distance from x_next to nearest sampled point ÷ ell."""
    x_next = np.asarray(x_next, dtype=float).reshape(1, -1)
    X_sampled = np.asarray(X_sampled, dtype=float)
    if X_sampled.size == 0:
        return np.nan
    dists = euclidean_distances(x_next, X_sampled).ravel()
    d_min = float(np.min(dists))
    if (ell is None) or (ell <= 0):
        return np.nan
    return d_min / ell

def _classify_strategy(sig_pct):
    """Assign strategy label from σ percentile (bounded exploration)."""
    if sig_pct <= 40:
        return "Exploit"
    elif sig_pct <= 75:
        return "Balanced"
    elif sig_pct <= 85:
        return "Explore"
    else:
        return "Too exploratory"

def sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled):
    """
    Builds a 1-row DataFrame with:
      μ_raw (raw), σ_pct (scaled %), dist_over_ℓ_raw (raw), acq_pct (scaled %), Strategy
    """

    # Ensure arrays are the correct shape
    X_candidates = np.asarray(X_candidates, dtype=float)
    acq_values = np.asarray(acq_values, dtype=float).ravel()
    x_next = np.asarray(x_next, dtype=float).ravel()

    # 1) Predict μ, σ for all candidates (for percentiles) and for x_next
    mu_all, sigma_all = model.predict(X_candidates, return_std=True)
    mu_xn, sigma_xn = model.predict(x_next.reshape(1, -1), return_std=True)
    mu_xn = float(mu_xn.ravel()[0])
    sigma_xn = float(sigma_xn.ravel()[0])

    # 2) Percentiles for σ and acquisition
    sig_pct = _percentile_of_score(sigma_all, sigma_xn)

    # Find acquisition value at x_next by matching to nearest candidate
    idx_closest = np.argmin(np.sum((X_candidates - x_next.reshape(1, -1))**2, axis=1))
    acq_xn = float(acq_values[idx_closest])
    acq_pct = _percentile_of_score(acq_values, acq_xn)

    # 3) Distance / ell
    ell = None
    if hasattr(model, "kernel_"):
        ell = _first_length_scale_from_kernel(model.kernel_)
    dist_over_ell = _nearest_dist_over_ell(x_next, X_sampled, ell)

    # 4) Strategy classification
    strategy = _classify_strategy(sig_pct)

    # 5) Build the 1-row DataFrame (rounded for readability)
    data = {
        "μ_raw (raw)":            np.round(mu_xn, 3),
        "σ_pct (scaled %)":       int(np.round(sig_pct)),
        "dist_over_ℓ_raw (raw)":  np.round(dist_over_ell, 3) if np.isfinite(dist_over_ell) else np.nan,
        "acq_pct (scaled %)":     int(np.round(acq_pct)),
        "Strategy":               strategy
    }

    return pd.DataFrame([data])
   

result_df = sanity_check_table(model, X_candidates, acq_values, x_next, y_best, X_sampled)
result_df

,μ_raw (raw),σ_pct (scaled %),dist_over_ℓ_raw (raw),acq_pct (scaled %),Strategy
0,-0.047,43,0.779,100,Balanced
